In [1]:
import numpy as np

logits = np.array([-2.0, -0.9, -0.5, 0.0, 0.8, 2.2])
truth  = np.array([0, 0, 1, 1, 1, 0])

p = 1 / (1 + np.exp(-logits))
print("Probabilities:", np.round(p, 4))

# 1. Logit that matches probability 0.3
logit_T03 = np.log(0.3 / 0.7)

# 2. Strict vs >= at T = 0.5
n_strict = int((p > 0.5).sum())
n_geq    = int((p >= 0.5).sum())

# 3. Dice score
def dice(pred, gt):
    tp = (pred * gt).sum()
    return 2 * tp / (pred.sum() + gt.sum())

d03 = dice((p > 0.3).astype(int), truth)
d05 = dice((p > 0.5).astype(int), truth)

print(f"{logit_T03:.4f}, {n_strict}, {n_geq}, {d03:.4f}, {d05:.4f}")

Probabilities: [0.1192 0.2891 0.3775 0.5    0.69   0.9002]
-0.8473, 2, 3, 0.8571, 0.4000


In [2]:
import numpy as np

configs = {
    "X": dict(fg=0.62, fp=0.9950, empty=0.40),
    "Y": dict(fg=0.58, fp=0.9990, empty=0.85),
    "Z": dict(fg=0.45, fp=0.9998, empty=0.97),
}

# Weights for (FgDice, FalsePositiveScore, EmptyAccuracy)
# Change these to match the competition's Evaluation page
W = (1/3, 1/3, 1/3)

scores = {}
for name, c in configs.items():
    s = W[0]*c["fg"] + W[1]*c["fp"] + W[2]*c["empty"]
    scores[name] = s
    print(f"Score_{name} = {s:.4f}")

best = max(scores, key=scores.get)
top_dice = max(configs, key=lambda k: configs[k]["fg"])
print(f"\nBest config: {best}  (highest FgDice was {top_dice})")
print(f"{scores['X']:.4f}, {scores['Y']:.4f}, {scores['Z']:.4f}, {best}, FalsePositiveScore and EmptyAccuracy")

Score_X = 0.6717
Score_Y = 0.8097
Score_Z = 0.8066

Best config: Y  (highest FgDice was X)
0.6717, 0.8097, 0.8066, Y, FalsePositiveScore and EmptyAccuracy


In [3]:
import numpy as np
from scipy import ndimage

mask = np.array([
    [1,1,0,0,0,0,0,1],
    [1,1,0,0,0,0,0,0],
    [0,0,1,0,0,0,0,0],
    [0,0,0,1,0,0,0,0],
    [0,0,0,0,0,1,1,0],
    [1,0,0,0,0,1,1,1],
])

# 8-connectivity: 3x3 block of ones
lab8, n8 = ndimage.label(mask, structure=np.ones((3, 3)))
# 4-connectivity: default cross shape
lab4, n4 = ndimage.label(mask)

sizes8 = np.bincount(lab8.ravel())[1:]   # skip background
print("8-conn component sizes:", sizes8)

pixels_N5 = int(sizes8[sizes8 >= 5].sum())
pixels_N6 = int(sizes8[sizes8 >= 6].sum())

print(f"{n8}, {n4}, {pixels_N5}, {pixels_N6}")

8-conn component sizes: [6 1 5 1]
4, 6, 11, 6


In [4]:
import numpy as np

CANVAS = 128 * 800   # 102,400 pixels

def score(fg, fp, ea):
    return 0.70*fg + 0.20*fp + 0.10*ea

# ---------- BEFORE filtering ----------
dice_before = [0.70]*12 + [0.45]*8
fp_px_before = [0]*50 + [20]*20 + [2000]*10      # predicted pixels on each empty pair

# ---------- AFTER filtering (min area N = 50) ----------
dice_after = [0.70]*12 + [0.45]*7 + [0.0]        # one 0.45 pair lost its only 30-px component
fp_px_after = [0]*50 + [0]*20 + [2000]*10        # speckles removed, blobs stay

def metrics(dice, fp_px):
    fg = np.mean(dice)
    fp = 1 - np.mean(np.array(fp_px) / CANVAS)
    ea = np.mean(np.array(fp_px) == 0)
    return fg, fp, ea

fg_b, fp_b, ea_b = metrics(dice_before, fp_px_before)
fg_a, fp_a, ea_a = metrics(dice_after, fp_px_after)

print(f"Before: FgDice={fg_b:.4f}  FP={fp_b:.4f}  EmptyAcc={ea_b:.4f}  Score={score(fg_b, fp_b, ea_b):.4f}")
print(f"After : FgDice={fg_a:.4f}  FP={fp_a:.4f}  EmptyAcc={ea_a:.4f}  Score={score(fg_a, fp_a, ea_a):.4f}")
print()
print(f"{fg_a:.4f}, {fp_a:.4f}, {ea_a:.4f}, {score(fg_b, fp_b, ea_b):.4f}, {score(fg_a, fp_a, ea_a):.4f}")

Before: FgDice=0.6000  FP=0.9975  EmptyAcc=0.6250  Score=0.6820
After : FgDice=0.5775  FP=0.9976  EmptyAcc=0.8750  Score=0.6913

0.5775, 0.9976, 0.8750, 0.6820, 0.6913


In [5]:
import torch

# shape (B, C, H, W) = (1, 1, 2, 2)
orig = torch.tensor([[0.9, 0.6], [0.4, 0.2]]).view(1, 1, 2, 2)
hf   = torch.tensor([[0.5, 0.8], [0.3, 0.4]]).view(1, 1, 2, 2)   # raw, in flipped frame
vf   = torch.tensor([[0.1, 0.5], [0.7, 0.9]]).view(1, 1, 2, 2)   # raw, in flipped frame

HFLIP_DIM, VFLIP_DIM = 3, 2   # W is dim 3, H is dim 2

# Correct: inverse-flip each augmented output, then average
correct = (orig + torch.flip(hf, dims=[HFLIP_DIM]) + torch.flip(vf, dims=[VFLIP_DIM])) / 3
# Buggy: average without un-flipping
buggy = (orig + hf + vf) / 3

print("Correct avg:\n", correct[0, 0])
print("Buggy avg:\n", buggy[0, 0])

top_right = correct[0, 0, 0, 1].item()
n_correct = int((correct > 0.5).sum())
n_buggy   = int((buggy > 0.5).sum())

print(f"{HFLIP_DIM}, {VFLIP_DIM}, {top_right:.4f}, {n_correct}, {n_buggy}")

Correct avg:
 tensor([[0.8000, 0.6667],
        [0.3000, 0.3333]])
Buggy avg:
 tensor([[0.5000, 0.6333],
        [0.4667, 0.5000]])
3, 2, 0.6667, 2, 1


In [6]:
import numpy as np

#              A     B     C
P = np.array([[0.90, 0.40, 0.45],   # p1
              [0.55, 0.55, 0.20],   # p2
              [0.80, 0.70, 0.90],   # p3
              [0.10, 0.20, 0.30],   # p4
              [0.60, 0.45, 0.55]])  # p5
y = np.array([1, 0, 1, 0, 1])
T = 0.5

def dice(pred, gt):
    pred = pred.astype(int)
    return 2 * (pred * gt).sum() / (pred.sum() + gt.sum())

# 1. Soft ensemble
soft = P.mean(axis=1)
dice_soft = dice(soft > T, y)

# 2. Hard vote (majority of 3)
votes = (P > T).sum(axis=1)
dice_vote = dice(votes >= 2, y)

# 3. Weighted ensemble at p1
w = np.array([0.5, 0.3, 0.2])
weighted_p1 = P[0] @ w

# 4. Probability-space vs logit-space averaging
sigmoid = lambda z: 1 / (1 + np.exp(-z))
logits = np.array([1.0, -3.0])
prob_avg  = sigmoid(logits).mean()
logit_avg = sigmoid(logits.mean())
method = "prob_avg" if prob_avg > 0.30 and logit_avg <= 0.30 else "logit_avg"

print("Soft averages:", np.round(soft, 4))
print("Votes:", votes)
print(f"{dice_soft:.4f}, {dice_vote:.4f}, {weighted_p1:.4f}, {prob_avg:.4f}, {logit_avg:.4f}, {method}")

Soft averages: [0.5833 0.4333 0.8    0.2    0.5333]
Votes: [1 2 3 0 2]
1.0000, 0.6667, 0.6600, 0.3892, 0.2689, prob_avg


In [7]:
import math

N_IMG, N_CLS = 5000, 4
H, W = 128, 800
PASS_MS = 12
GB = 1e9

values = N_IMG * N_CLS * H * W

# 1. One full set of probability maps
gb_f32 = values * 4 / GB
gb_f16 = values * 2 / GB
gb_u8  = values * 1 / GB

# 2. Naive memory (every model x view kept as float32)
sets_A, sets_B = 3 * 3, 2 * 2
gb_A = sets_A * gb_f32
gb_B = sets_B * gb_f32
fitA, fitB = gb_A <= 16, gb_B <= 16
fits = "both" if fitA and fitB else "A" if fitA else "B" if fitB else "neither"
accumulator_gb = gb_f32

# 3. Forward passes and minutes
passes_A = N_IMG * sets_A
passes_B = N_IMG * sets_B
min_A = passes_A * PASS_MS / 1000 / 60
min_B = passes_B * PASS_MS / 1000 / 60

# 4. Max whole models within 30 - 8 = 22 minutes
budget_s = (30 - 8) * 60
per_model_3view = N_IMG * 3 * PASS_MS / 1000
per_model_2view = N_IMG * 2 * PASS_MS / 1000
max3 = math.floor(budget_s / per_model_3view + 1e-9)
max2 = math.floor(budget_s / per_model_2view + 1e-9)

print(f"{gb_f32:.3f}, {gb_f16:.3f}, {gb_u8:.3f}, {gb_A:.3f}, {gb_B:.3f}, {fits}, "
      f"{accumulator_gb:.3f}, {passes_A}, {passes_B}, {min_A:.1f}, {min_B:.1f}, {max3}, {max2}")

8.192, 4.096, 2.048, 73.728, 32.768, neither, 8.192, 45000, 20000, 9.0, 4.0, 7, 11
